# ZTF reference catalogs vs Pan-STARRS: matching and filtering, quadrant 643_11_1

ZTF side: IRSA's DAOPhot PSF catalogs of the three deep reference images (`refpsfcat`), merged into objects by mutual nearest neighbour at 1". PS1 side: the DR2 stack-object catalog for the quadrant. The two are matched the same way. Tables were built on 2026-11-11 and live in `Data/ztf_data/`.

## 1. Setup

In [ ]:
import glob, os, sys
sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.spatial import cKDTree
from scipy.ndimage import distance_transform_edt
from astropy.nddata import Cutout2D
from astropy.coordinates import SkyCoord
from astropy.visualization import simple_norm
import astropy.units as u
from Extracting.images import load_ztf_refimg, load_ps1_skycell
from Extracting.ztf_images import refimg_path

D = '../Data'
obj = pd.read_parquet(f'{D}/ztf_data/ztfobj_643_11_1.parquet')      # one row per merged ZTF object
ps1 = pd.read_parquet(f'{D}/ztf_data/ps1_643_11_1.parquet')         # PS1 objects on valid ZTF pixels; obj.ps1_idx indexes this
ztf_cat = {b: pd.read_parquet(f'{D}/ztf_data/ztfpsf_643_11_1_{b}.parquet') for b in 'gri'}   # per-band DAOPhot catalogs
cosd = np.cos(np.radians(ps1.dec.median()))

obj['matched'] = obj.ps1_idx >= 0
ps1['matched'] = np.isin(np.arange(len(ps1)), obj.ps1_idx[obj.matched])
for b in 'gri':   # per-band DAOPhot flags and PS1 photometry on the object table
    obj[f'{b}_flags'] = np.where(obj[f'{b}_idx'] >= 0, ztf_cat[b]['flags'].values[obj[f'{b}_idx'].clip(lower=0)], 0)
    obj[f'ps1_{b}_psf'] = np.where(obj.matched, ps1[f'{b}PSFMag'].values[obj.ps1_idx.clip(lower=0)], np.nan)
    obj[f'ps1_{b}_kron'] = np.where(obj.matched, ps1[f'{b}KronMag'].values[obj.ps1_idx.clip(lower=0)], np.nan)
print(f'{len(obj)} ZTF objects, {obj.matched.sum()} matched; {len(ps1)} PS1 objects, {(~ps1.matched).sum()} PS1-only')

In [ ]:
# Images, masks and cutout helpers.
ztf_imgs = {b: load_ztf_refimg(refimg_path(f'{D}/ztf_data', 643, 11, 1, b)) for b in 'gri'}
nan_dist = {b: distance_transform_edt(~im.bad) for b, im in ztf_imgs.items()}   # px to the nearest masked (NaN) pixel

def min_nan_distance(ra, dec):
    """Smallest distance to a masked ZTF pixel over the three bands, in pixels; inf outside every image."""
    out = np.full(len(ra), np.inf)
    for b, im in ztf_imgs.items():
        x, y = im.wcs.world_to_pixel_values(ra, dec)
        inside = (x >= 0) & (x < im.shape[1]) & (y >= 0) & (y < im.shape[0])
        xi, yi = np.clip(x.round().astype(int), 0, im.shape[1] - 1), np.clip(y.round().astype(int), 0, im.shape[0] - 1)
        out = np.where(inside, np.minimum(out, nan_dist[b][yi, xi]), out)
    return out

_ps1_imgs = {}
def ps1_image_at(ra, dec):
    """The local PS1 r skycell containing the position (loaded on first use), or None."""
    for stack in sorted(glob.glob(f'{D}/ps1_data/rings.v3.skycell.*.stk.r.unconv.fits')):
        name = os.path.basename(stack).split('.stk.')[0]
        if name not in _ps1_imgs:
            _ps1_imgs[name] = load_ps1_skycell(stack, stack.replace('.fits', '.mask.fits'))
        im = _ps1_imgs[name]
        x, y = im.wcs.world_to_pixel_values(ra, dec)
        if 70 < x < im.shape[1] - 70 and 70 < y < im.shape[0] - 70:
            return im
    return None

MARKERS = {'matched': 'white', 'ZTF only': 'red', 'PS1 only': 'orange'}
def _marker_sets():
    return {'matched': (obj.ra[obj.matched].values, obj.dec[obj.matched].values),
            'ZTF only': (obj.ra[~obj.matched].values, obj.dec[~obj.matched].values),
            'PS1 only': (ps1.ra[~ps1.matched].values, ps1.dec[~ps1.matched].values)}
HANDLES = [Line2D([], [], marker='o', ls='', markerfacecolor='none', markeredgecolor=c, markersize=9, label=l) for l, c in MARKERS.items()]

def north_up_east_left(cut):
    """Flip a cutout to north-up, east-left (ZTF references are stored north-down, east-right) and return a coordinate mapper."""
    cd = cut.wcs.pixel_scale_matrix
    flip_x, flip_y = cd[0, 0] > 0, cd[1, 1] < 0
    data = cut.data[:, ::-1] if flip_x else cut.data
    data = data[::-1, :] if flip_y else data
    ny, nx = cut.shape
    return data, lambda x, y: ((nx - 1 - x if flip_x else x), (ny - 1 - y if flip_y else y))

def show(ax, im, ra, dec, size_arcsec=30):
    """One cutout with every object drawn in its match-status colour and a cross at (ra, dec)."""
    cut = Cutout2D(im.data, SkyCoord(ra * u.deg, dec * u.deg), size_arcsec * u.arcsec, wcs=im.wcs)
    data, to_display = north_up_east_left(cut)
    if np.isfinite(data).any():
        ax.imshow(data, origin='lower', cmap='viridis', norm=simple_norm(data, 'log', percent=99.5))
    else:
        ax.imshow(np.zeros_like(data), origin='lower', cmap='gray'); ax.text(0.5, 0.5, 'no data', color='w', ha='center', transform=ax.transAxes)
    for label, (cra, cdec) in _marker_sets().items():
        x, y = to_display(*cut.wcs.world_to_pixel_values(cra, cdec))
        s = (x > 0) & (x < cut.shape[1]) & (y > 0) & (y < cut.shape[0])
        ax.scatter(x[s], y[s], s=70, facecolors='none', edgecolors=MARKERS[label], lw=1.3)
    x0, y0 = to_display(*cut.wcs.world_to_pixel_values(ra, dec))
    ax.plot(x0, y0, '+', color='white', ms=16, mew=1.5)
    ax.set_xticks([]); ax.set_yticks([])

def cutout_grid(rows, ps1_row=True, title=None):
    """`rows` = [(ra, dec, label), ...]; one column per object, ZTF r on top and, optionally, PS1 r below."""
    n = len(rows); nr = 2 if ps1_row else 1
    fig, axes = plt.subplots(nr, n, figsize=(3.4 * n, 3.6 * nr), squeeze=False)
    for j, (ra, dec, label) in enumerate(rows):
        show(axes[0, j], ztf_imgs['r'], ra, dec); axes[0, j].set_title(label, fontsize=8)
        if ps1_row:
            pim = ps1_image_at(ra, dec)
            show(axes[1, j], pim, ra, dec) if pim is not None else axes[1, j].set_axis_off()
    axes[0, 0].set_ylabel('ZTF r reference'); axes[-1, 0].set_ylabel('PS1 r stack' if ps1_row else 'ZTF r reference')
    axes[0, -1].legend(handles=HANDLES, loc='upper right', frameon=True, facecolor='0.3', labelcolor='white', fontsize=7)
    if title: fig.suptitle(title, fontsize=10)
    plt.tight_layout(); plt.show()

def waterfall(df, filters, extra=None, extra_label=''):
    """Apply `filters` ({label: fn(df) -> keep mask}) cumulatively; print survivors; return {label: cumulative mask}."""
    ok = np.ones(len(df), bool); masks = {'all': ok.copy()}
    print(f'{"step":48s} {"objects":>8s} {extra_label:>10s}')
    print(f'{"all":48s} {len(df):8d} {int(extra.sum()) if extra is not None else "":>10}')
    for label, fn in filters.items():
        ok = ok & np.asarray(fn(df), bool); masks[label] = ok.copy()
        print(f'{label:48s} {ok.sum():8d} {int((ok & extra).sum()) if extra is not None else "":>10}')
    return masks

def removed_examples(df, filters, label, n=4, seed=0):
    """Objects removed by `filters[label]` among those passing every earlier filter."""
    ok = np.ones(len(df), bool)
    for lab, fn in filters.items():
        if lab == label: break
        ok &= np.asarray(fn(df), bool)
    gone = df[ok & ~np.asarray(filters[label](df), bool)]
    return gone.sample(min(n, len(gone)), random_state=seed) if len(gone) else gone

def mag_hist(df, masks, mag, xlabel, bins=np.arange(12, 25, 0.5)):
    fig, ax = plt.subplots(figsize=(8, 4.2))
    for k, (label, m) in enumerate(masks.items()):
        ax.hist(mag[m].dropna(), bins=bins, histtype='step', lw=1.5, color=f'C{k % 10}', label=f'{label} ({m.sum()})')
    ax.set_xlabel(xlabel); ax.set_ylabel('objects'); ax.set_yscale('log'); ax.legend(frameon=False, fontsize=8)
    plt.tight_layout(); plt.show()

## 2. The match

In [ ]:
matched = obj[obj.matched]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
bins = np.linspace(0, 1, 51)
axes[0].hist(matched.ps1_sep, bins=bins, histtype='step', lw=1.5, color='k', label='all matches')
axes[0].hist(matched.ps1_sep[matched.best_snr > 20], bins=bins, histtype='step', lw=1.5, color='C3', label='best-band SNR > 20')
axes[0].set_xlabel('ZTF object to PS1 object separation [arcsec]'); axes[0].set_ylabel('objects'); axes[0].legend(frameon=False)
for b, c in zip('rg', ['C0', 'C2']):
    s = obj[f'{b}_sep'].dropna()
    axes[1].hist(s, bins=bins, histtype='step', lw=1.5, color=c, label=f'{b} source to i-seeded object (median {s.median():.2f}")')
axes[1].set_xlabel('band-to-object separation within ZTF [arcsec]'); axes[1].set_ylabel('objects'); axes[1].legend(frameon=False)
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharex='col', gridspec_kw={'height_ratios': [2, 1]})
for j, b in enumerate('gri'):
    m = matched[matched[f'{b}_idx'] >= 0]
    x, y = m[f'ps1_{b}_psf'], m[f'{b}_mag']; star = (m[f'ps1_{b}_psf'] - m[f'ps1_{b}_kron']).abs() < 0.05
    lim = (13, 24.5)
    axes[0, j].scatter(x[~star], y[~star], s=2, alpha=0.3, color='C1', label='extended / other')
    axes[0, j].scatter(x[star], y[star], s=2, alpha=0.5, color='k', label='point sources')
    axes[0, j].plot(lim, lim, 'k--', lw=0.8); axes[0, j].set_xlim(lim); axes[0, j].set_ylim(lim)
    axes[0, j].set_ylabel(f'ZTF {b} PSF mag'); axes[0, j].legend(frameon=False, loc='upper left', markerscale=5)
    d = y - x
    axes[1, j].scatter(x[~star], d[~star], s=2, alpha=0.3, color='C1'); axes[1, j].scatter(x[star], d[star], s=2, alpha=0.5, color='k')
    axes[1, j].axhline(0, color='k', lw=0.8, ls='--'); axes[1, j].set_ylim(-1.5, 1.5)
    axes[1, j].set_xlabel(f'PS1 {b} PSF mag'); axes[1, j].set_ylabel('ZTF − PS1')
    sel = star & (x > 16) & (x < 20)
    axes[1, j].text(0.02, 0.05, f'stars 16–20: median {d[sel].median():+.3f}, MAD {1.4826*np.median(np.abs(d[sel]-d[sel].median())):.3f}', transform=axes[1, j].transAxes, fontsize=9)
plt.tight_layout(); plt.show()

## 3. The filters

Each filter is a function of a table returning a keep-mask. ZTF filters read the DAOPhot columns of the merged object table; PS1 filters read the DR2 flag columns. Bit values are from the archive's flag pages. Variability, transient, QSO and extended-source bits are not cut here; they are kept for after filtering.

In [ ]:
SNR_MIN = 5.0
NAN_RADIUS_PS1_PX = 5                                  # original association: PS1 sources within 5 px of a masked ZTF pixel were dropped
BAD_STACK, SUSPECT_STACK = 128, 64                     # qualityFlag
DEFECT, PEAK_ON_SPIKE, SATSTAR = 2048, 1 << 29, 128     # infoFlag
ON_STARCORE, ON_BURNTOOL, SATSTAR_PROFILE = 16, 32, 4096   # infoFlag2

def has_bit(col, bit):
    return (col.fillna(0).astype('int64') & bit) != 0

# ---- ZTF (per detected band) ----
def ztf_flags_ok(df):
    """DAOPhot flags == 0 in every detected band (the only non-zero value seen is 1: masked pixels in the 5x5 box)."""
    return ~np.any([(df[f'{b}_idx'] >= 0) & (df[f'{b}_flags'] != 0) for b in 'gri'], axis=0)

def ztf_n_snr_bands(df):
    return sum(((df[f'{b}_snr'] >= SNR_MIN).fillna(False)).astype(int) for b in 'gri')

def ztf_snr_1band(df): return (ztf_n_snr_bands(df) >= 1).values
def ztf_snr_2bands(df): return (ztf_n_snr_bands(df) >= 2).values

ZTF_FILTERS = {'flags == 0 in every detected band': ztf_flags_ok,
               f'SNR >= {SNR_MIN:.0f} in >= 1 band': ztf_snr_1band,
               f'SNR >= {SNR_MIN:.0f} in >= 2 bands': ztf_snr_2bands}

# ---- PS1 (per measured band) ----
def _ps1_any_band(df, flag_col, bits):
    out = np.zeros(len(df), bool)
    for b in 'gri':
        measured = df[f'{b}PSFMag'].notna()
        for bit in bits:
            out |= measured & has_bit(df[f'{b}{flag_col}'], bit)
    return out

def ps1_not_near_masked(df): return min_nan_distance(df.ra.values, df.dec.values) > NAN_RADIUS_PS1_PX
def ps1_stack_ok(df): return ~(has_bit(df.qualityFlag, BAD_STACK) | has_bit(df.qualityFlag, SUSPECT_STACK)).values
def ps1_no_defect(df): return ~_ps1_any_band(df, 'infoFlag', [DEFECT])
def ps1_no_spike(df): return ~_ps1_any_band(df, 'infoFlag', [PEAK_ON_SPIKE])
def ps1_no_starcore(df): return ~_ps1_any_band(df, 'infoFlag2', [ON_STARCORE, ON_BURNTOOL])
def ps1_not_saturated(df): return ~(_ps1_any_band(df, 'infoFlag', [SATSTAR]) | _ps1_any_band(df, 'infoFlag2', [SATSTAR_PROFILE]))

def ps1_n_snr_bands(df):
    return pd.DataFrame({b: 1.0857 / df[f'{b}PSFMagErr'] >= SNR_MIN for b in 'gri'}).sum(axis=1)

def ps1_snr_1band(df): return (ps1_n_snr_bands(df) >= 1).values
def ps1_snr_2bands(df): return (ps1_n_snr_bands(df) >= 2).values

PS1_FILTERS = {f'not within {NAN_RADIUS_PS1_PX} px of a masked ZTF pixel': ps1_not_near_masked,
               'not BAD_STACK / SUSPECT_STACK': ps1_stack_ok,
               'not DEFECT': ps1_no_defect,
               'not PEAK_ON_SPIKE': ps1_no_spike,
               'not ON_STARCORE / ON_BURNTOOL': ps1_no_starcore,
               'not SATSTAR / SATSTAR_PROFILE': ps1_not_saturated,
               f'SNR >= {SNR_MIN:.0f} in >= 1 band': ps1_snr_1band,
               f'SNR >= {SNR_MIN:.0f} in >= 2 bands': ps1_snr_2bands}

## 4. PS1-only objects

### 4.1 Waterfall

In [ ]:
ps1_only = ps1[~ps1.matched].copy()
ps1_masks = waterfall(ps1_only, PS1_FILTERS, extra=(ps1_only.rPSFMag < 19).values, extra_label='r<19')
ps1_only_clean = ps1_only[ps1_masks[f'SNR >= {SNR_MIN:.0f} in >= 1 band']]    # the 2-band line is shown for reference only
mag_hist(ps1_only, ps1_masks, ps1_only.rPSFMag, 'PS1 r PSF mag')

### 4.2 What each cut removes

Four objects at random removed by each cut, among those that passed the cuts before it. ZTF r on top, PS1 r below; the cross is the object.

In [ ]:
for label in list(PS1_FILTERS)[:-2]:   # the SNR cuts remove faint objects by construction
    ex = removed_examples(ps1_only, PS1_FILTERS, label)
    if len(ex) == 0:
        print(f'{label}: removes nothing beyond the earlier cuts'); continue
    cutout_grid([(r.ra, r.dec, f'r={r.rPSFMag:.1f}  qf={int(r.qualityFlag)}') for r in ex.itertuples()], title=f'removed by: {label}')

### 4.3 Are the survivors blend components?

Distance from each surviving PS1-only object to the nearest other PS1 object, and whether that neighbour is the one ZTF matched.

In [ ]:
surv = ps1_only_clean
d, i = cKDTree(np.c_[ps1.ra.values * cosd, ps1.dec.values]).query(np.c_[surv.ra.values * cosd, surv.dec.values], k=2)
d_other, j_other = d[:, 1] * 3600, i[:, 1]
neighbour_matched = ps1.matched.values[j_other]
print(f'{"radius":>8s} {"has neighbour":>14s} {"neighbour matched":>18s} {"r<19 with neighbour":>20s}')
for r in (1.0, 2.0, 3.0, 5.0):
    near = d_other < r
    print(f'{r:7.1f}" {near.sum():8d} ({100*near.mean():3.0f}%) {(near & neighbour_matched).sum():18d} {(near & (surv.rPSFMag < 19).values).sum():12d} / {(surv.rPSFMag < 19).sum()}')
fig, ax = plt.subplots(figsize=(8, 4.2))
bins = np.arange(0, 10.25, 0.25)
ax.hist(d_other[neighbour_matched], bins=bins, histtype='stepfilled', alpha=0.5, color='C0', label='neighbour matched to ZTF')
ax.hist(d_other[~neighbour_matched], bins=bins, histtype='step', lw=1.5, color='C3', label='neighbour not matched')
ax.set_xlabel('distance to nearest other PS1 object [arcsec]'); ax.set_ylabel('surviving PS1-only objects'); ax.legend(frameon=False)
plt.tight_layout(); plt.show()

## 5. ZTF-only objects

### 5.1 Waterfall

In [ ]:
ztf_only = obj[~obj.matched].copy()
ztf_masks = waterfall(ztf_only, ZTF_FILTERS, extra=(ztf_only.best_snr > 10).values, extra_label='bestSNR>10')
ztf_only_clean = ztf_only[ztf_masks[f'SNR >= {SNR_MIN:.0f} in >= 2 bands']]
ztf_only['best_mag'] = ztf_only[['g_mag', 'r_mag', 'i_mag']].min(axis=1)
mag_hist(ztf_only, ztf_masks, ztf_only.best_mag, 'brightest-band ZTF PSF mag', bins=np.arange(14, 25, 0.5))

### 5.2 What each cut removes

ZTF r only.

In [ ]:
for label in ZTF_FILTERS:
    ex = removed_examples(ztf_only, ZTF_FILTERS, label)
    if len(ex) == 0:
        print(f'{label}: removes nothing beyond the earlier cuts'); continue
    cutout_grid([(r.ra, r.dec, f'nbands={r.nbands}  SNR={r.best_snr:.0f}') for r in ex.itertuples()], ps1_row=False, title=f'removed by: {label}')

## 6. Survivors

The four brightest and four random objects left on each side after filtering.

In [ ]:
surv = ps1_only_clean[ps1_only_clean.rPSFMag.notna()]
top = surv.sort_values('rPSFMag').head(4); rest = surv.drop(top.index).sample(4, random_state=0)
cutout_grid([(r.ra, r.dec, f'r={r.rPSFMag:.1f}  Kron={r.rKronMag:.1f}') for r in pd.concat([top, rest]).itertuples()], title='PS1-only survivors: 4 brightest, 4 random')
surv = ztf_only_clean.copy(); surv['best_mag'] = surv[['g_mag', 'r_mag', 'i_mag']].min(axis=1)
top = surv.sort_values('best_mag').head(4); rest = surv.drop(top.index).sample(4, random_state=0)
cutout_grid([(r.ra, r.dec, f'mag={r.best_mag:.1f}  nbands={r.nbands}  SNR={r.best_snr:.0f}') for r in pd.concat([top, rest]).itertuples()], title='ZTF-only survivors: 4 brightest, 4 random')